# Base vs Trained Benchmark

This notebook runs the SWI-Prolog benchmark for the base model and the fine-tuned model, then compares their results.

In [1]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

NOTEBOOK_ROOT = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == "src" else NOTEBOOK_ROOT
SRC_ROOT = PROJECT_ROOT / "src"

for import_root in (PROJECT_ROOT, SRC_ROOT):
    if str(import_root) not in sys.path:
        sys.path.append(str(import_root))

from src.benchmark_tasks import load_benchmark_tasks
from src.eval_runner import check_swipl_exists, evaluate_task
from src.generate_solutions import build_model_prompt, extract_code


c:\Projects\TgSummarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def env_flag(name: str, default: bool = False) -> bool:
    raw_value = os.getenv(name)
    if raw_value is None:
        return default
    return raw_value.strip().lower() in {"1", "true", "yes", "on"}


BASE_MODEL_ID = os.getenv("TRAIN_BASE_MODEL_ID")
TRAINED_MODEL_PATH = os.getenv(
    "TRAINED_MODEL_PATH",
    str(PROJECT_ROOT / "src" / "prolog_merged"),
)
TASKS_PATH = Path(os.getenv("BENCHMARK_TASKS_PATH", str(PROJECT_ROOT / "benchmark" / "tasks_swipl.jsonl")))
OUTPUT_DIR = Path(os.getenv("MODEL_BENCHMARK_OUTPUT_DIR", str(PROJECT_ROOT / "outputs" / "model_benchmark")))

TASK_LIMIT = int(os.getenv("BENCHMARK_TASK_LIMIT", "0"))
TASK_IDS = [task_id.strip() for task_id in os.getenv("BENCHMARK_TASK_IDS", "").split(",") if task_id.strip()]
MAX_NEW_TOKENS = int(os.getenv("BENCHMARK_MAX_NEW_TOKENS", "256"))
USE_4BIT = env_flag("BENCHMARK_LOAD_IN_4BIT", default=True)
USE_8BIT = env_flag("BENCHMARK_LOAD_IN_8BIT", default=False)
TEMPERATURE = float(os.getenv("BENCHMARK_TEMPERATURE", "0.0"))
DO_SAMPLE = env_flag("BENCHMARK_DO_SAMPLE", default=False)

if not BASE_MODEL_ID:
    raise RuntimeError("TRAIN_BASE_MODEL_ID is not set.")
if USE_4BIT and USE_8BIT:
    raise RuntimeError("BENCHMARK_LOAD_IN_4BIT and BENCHMARK_LOAD_IN_8BIT cannot both be enabled.")

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else torch.float16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "TRAINED_MODEL_PATH": TRAINED_MODEL_PATH,
    "TASKS_PATH": str(TASKS_PATH),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "TASK_LIMIT": TASK_LIMIT,
    "TASK_IDS": TASK_IDS,
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "USE_4BIT": USE_4BIT,
    "USE_8BIT": USE_8BIT,
    "TEMPERATURE": TEMPERATURE,
    "DO_SAMPLE": DO_SAMPLE,
    "device": device,
    "dtype": str(model_dtype),
})

{'BASE_MODEL_ID': 'Qwen/Qwen3.5-2B', 'TRAINED_MODEL_PATH': 'C:\\Projects\\TgSummarizer\\src\\prolog_merged', 'TASKS_PATH': 'C:\\Projects\\TgSummarizer\\benchmark\\tasks_swipl.jsonl', 'OUTPUT_DIR': 'C:\\Projects\\TgSummarizer\\outputs\\model_benchmark', 'TASK_LIMIT': 0, 'TASK_IDS': [], 'MAX_NEW_TOKENS': 256, 'USE_4BIT': True, 'USE_8BIT': False, 'TEMPERATURE': 0.0, 'DO_SAMPLE': False, 'device': 'cuda', 'dtype': 'torch.bfloat16'}


In [3]:
check_swipl_exists()
tasks = load_benchmark_tasks(TASKS_PATH)

if TASK_IDS:
    wanted = set(TASK_IDS)
    tasks = [task for task in tasks if task.task_id in wanted]

if TASK_LIMIT > 0:
    tasks = tasks[:TASK_LIMIT]

if not tasks:
    raise RuntimeError("No benchmark tasks selected.")

print(f"Selected tasks: {len(tasks)}")
print([task.task_id for task in tasks[:10]])

Selected tasks: 10
['family_grandparent', 'my_member', 'list_last', 'factorial', 'fibonacci', 'path_graph', 'sum_list', 'clpfd_simple', 'dcg_digits', 'reverse_list']


In [4]:
def build_quantization_config():
    if not device.startswith("cuda"):
        return None
    if not USE_4BIT and not USE_8BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=USE_4BIT,
        load_in_8bit=USE_8BIT,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=model_dtype,
    )


def load_model_bundle(model_ref: str):
    tokenizer = AutoTokenizer.from_pretrained(model_ref, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "trust_remote_code": True,
        "dtype": model_dtype,
    }
    quantization_config = build_quantization_config()
    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config
        model_kwargs["device_map"] = "auto"
    print('Tokenizer loaded')
    model = AutoModelForCausalLM.from_pretrained(model_ref, **model_kwargs)
    if quantization_config is None:
        model.to(device)

    model.eval()
    model.config.pad_token_id = tokenizer.pad_token_id
    if hasattr(model, "generation_config"):
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    return tokenizer, model


def get_input_device(model):
    hf_device_map = getattr(model, "hf_device_map", None)
    if isinstance(hf_device_map, dict):
        for mapped_device in hf_device_map.values():
            mapped = str(mapped_device)
            if mapped not in {"cpu", "disk", "meta"}:
                return mapped
        if hf_device_map:
            return str(next(iter(hf_device_map.values())))

    return str(getattr(model, "device", device))


def generate_solution(tokenizer, model, task, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": "You write only SWI-Prolog code without explanations."},
        {"role": "user", "content": build_model_prompt(task)},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        template_kwargs = {
            "tokenize": True,
            "add_generation_prompt": True,
            "return_tensors": "pt",
            "return_dict": True,
        }
        try:
            inputs = tokenizer.apply_chat_template(
                messages,
                enable_thinking=False,
                **template_kwargs,
            )
        except TypeError:
            inputs = tokenizer.apply_chat_template(messages, **template_kwargs)
    else:
        prompt = "\n\n".join(message["content"] for message in messages)
        inputs = tokenizer(prompt, return_tensors="pt")

    input_device = get_input_device(model)
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if DO_SAMPLE:
        generation_kwargs["temperature"] = TEMPERATURE

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generation_kwargs)

    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, prompt_length:]
    raw_text = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return extract_code(raw_text)


def unload_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [5]:
def run_model_benchmark(model_name: str, model_ref: str, selected_tasks):
    print(f"Loading {model_name}: {model_ref}")
    tokenizer, model = load_model_bundle(model_ref)
    results = []
    started_at = time.time()

    try:
        for index, task in enumerate(selected_tasks, start=1):
            print(f"[{model_name}] {index}/{len(selected_tasks)} {task.task_id}")
            solution_text = generate_solution(tokenizer, model, task)
            evaluation = evaluate_task(task, solution_text)
            evaluation["task_id"] = task.task_id
            evaluation["solution_text"] = solution_text
            results.append(evaluation)
    finally:
        del tokenizer
        unload_model(model)

    elapsed_sec = time.time() - started_at
    total = len(results)
    syntax_ok = sum(1 for result in results if result.get("syntax_ok"))
    passed_all = sum(1 for result in results if result.get("all_passed"))
    total_timeouts = sum(int(result.get("timeouts", 0)) for result in results)
    total_tests = sum(int(result.get("total_tests", 0)) for result in results)

    summary = {
        "model_name": model_name,
        "model_ref": model_ref,
        "tasks": total,
        "pass_rate": (passed_all / total) if total else 0.0,
        "syntax_pass_rate": (syntax_ok / total) if total else 0.0,
        "timeout_rate": (total_timeouts / total_tests) if total_tests else 0.0,
        "passed_all": passed_all,
        "syntax_ok": syntax_ok,
        "total_timeouts": total_timeouts,
        "total_tests": total_tests,
        "elapsed_sec": elapsed_sec,
    }
    return summary, results

In [6]:
base_summary, base_results = run_model_benchmark("base", BASE_MODEL_ID, tasks)
base_summary

Loading base: Qwen/Qwen3.5-2B
Tokenizer loaded


W0607 16:06:51.997000 14340 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   0%|          | 1/320 [00:01<05:19,  1.00s/it]c:\Projects\TgSummarizer\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 320/320 [00:02<00:00, 112.81it/s]


[base] 1/10 family_grandparent


c:\Projects\TgSummarizer\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[base] 2/10 my_member
[base] 3/10 list_last
[base] 4/10 factorial
[base] 5/10 fibonacci
[base] 6/10 path_graph
[base] 7/10 sum_list
[base] 8/10 clpfd_simple
[base] 9/10 dcg_digits
[base] 10/10 reverse_list


{'model_name': 'base',
 'model_ref': 'Qwen/Qwen3.5-2B',
 'tasks': 10,
 'pass_rate': 0.0,
 'syntax_pass_rate': 0.9,
 'timeout_rate': 0.0,
 'passed_all': 0,
 'syntax_ok': 9,
 'total_timeouts': 0,
 'total_tests': 20,
 'elapsed_sec': 66.28961491584778}

In [7]:
base_failed_results = [result for result in base_results if not result["all_passed"]]
base_failed_task_ids = [result["task_id"] for result in base_failed_results]

print(f"Base failed tasks: {len(base_failed_results)}")
for result in base_failed_results:
    print(
        {
            "task_id": result["task_id"],
            "syntax_ok": result["syntax_ok"],
            "tests": f"{result['passed_tests']}/{result['total_tests']}",
            "timeouts": result.get("timeouts", 0),
        }
    )

Base failed tasks: 10
{'task_id': 'family_grandparent', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}
{'task_id': 'my_member', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}
{'task_id': 'list_last', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'factorial', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'fibonacci', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'path_graph', 'syntax_ok': False, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'sum_list', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'clpfd_simple', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'dcg_digits', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'reverse_list', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}


In [8]:
trained_summary, trained_results = run_model_benchmark("trained", TRAINED_MODEL_PATH, tasks)
trained_summary

Loading trained: C:\Projects\TgSummarizer\src\prolog_merged
Tokenizer loaded


Loading weights: 100%|██████████| 320/320 [00:02<00:00, 107.47it/s]


[trained] 1/10 family_grandparent
[trained] 2/10 my_member
[trained] 3/10 list_last
[trained] 4/10 factorial
[trained] 5/10 fibonacci
[trained] 6/10 path_graph
[trained] 7/10 sum_list
[trained] 8/10 clpfd_simple
[trained] 9/10 dcg_digits
[trained] 10/10 reverse_list


{'model_name': 'trained',
 'model_ref': 'C:\\Projects\\TgSummarizer\\src\\prolog_merged',
 'tasks': 10,
 'pass_rate': 0.2,
 'syntax_pass_rate': 1.0,
 'timeout_rate': 0.0,
 'passed_all': 2,
 'syntax_ok': 10,
 'total_timeouts': 0,
 'total_tests': 20,
 'elapsed_sec': 28.317045211791992}

In [9]:
trained_failed_results = [result for result in trained_results if not result["all_passed"]]
trained_failed_task_ids = [result["task_id"] for result in trained_failed_results]

print(f"Trained failed tasks: {len(trained_failed_results)}")
for result in trained_failed_results:
    print(
        {
            "task_id": result["task_id"],
            "syntax_ok": result["syntax_ok"],
            "tests": f"{result['passed_tests']}/{result['total_tests']}",
            "timeouts": result.get("timeouts", 0),
        }
    )

Trained failed tasks: 8
{'task_id': 'family_grandparent', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}
{'task_id': 'my_member', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'list_last', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'factorial', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}
{'task_id': 'fibonacci', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}
{'task_id': 'path_graph', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'dcg_digits', 'syntax_ok': True, 'tests': '0/2', 'timeouts': 0}
{'task_id': 'reverse_list', 'syntax_ok': True, 'tests': '1/2', 'timeouts': 0}


In [10]:
base_only_failures = []
for task in tasks:
    base_result = next(result for result in base_results if result["task_id"] == task.task_id)
    trained_result = next(result for result in trained_results if result["task_id"] == task.task_id)
    if (not base_result["all_passed"]) and trained_result["all_passed"]:
        base_only_failures.append(
            {
                "task_id": task.task_id,
                "base_syntax_ok": base_result["syntax_ok"],
                "base_tests": f"{base_result['passed_tests']}/{base_result['total_tests']}",
                "trained_tests": f"{trained_result['passed_tests']}/{trained_result['total_tests']}",
                "base_solution": base_result["solution_text"],
                "trained_solution": trained_result["solution_text"],
            }
        )

print(f"Tasks where base failed but trained passed: {len(base_only_failures)}")
for row in base_only_failures:
    print(
        {
            "task_id": row["task_id"],
            "base_syntax_ok": row["base_syntax_ok"],
            "base_tests": row["base_tests"],
            "trained_tests": row["trained_tests"],
        }
    )

Tasks where base failed but trained passed: 2
{'task_id': 'sum_list', 'base_syntax_ok': True, 'base_tests': '0/2', 'trained_tests': '2/2'}
{'task_id': 'clpfd_simple', 'base_syntax_ok': True, 'base_tests': '0/2', 'trained_tests': '2/2'}


In [11]:
comparison = {
    "base": base_summary,
    "trained": trained_summary,
}

per_task = []
base_by_task = {result["task_id"]: result for result in base_results}
trained_by_task = {result["task_id"]: result for result in trained_results}

for task in tasks:
    base_result = base_by_task[task.task_id]
    trained_result = trained_by_task[task.task_id]
    per_task.append(
        {
            "task_id": task.task_id,
            "base_pass": base_result["all_passed"],
            "trained_pass": trained_result["all_passed"],
            "base_tests": f"{base_result['passed_tests']}/{base_result['total_tests']}",
            "trained_tests": f"{trained_result['passed_tests']}/{trained_result['total_tests']}",
            "base_syntax": base_result["syntax_ok"],
            "trained_syntax": trained_result["syntax_ok"],
        }
    )

print(json.dumps(comparison, indent=2, ensure_ascii=False))
print("")
for row in per_task:
    print(row)

{
  "base": {
    "model_name": "base",
    "model_ref": "Qwen/Qwen3.5-2B",
    "tasks": 10,
    "pass_rate": 0.0,
    "syntax_pass_rate": 0.9,
    "timeout_rate": 0.0,
    "passed_all": 0,
    "syntax_ok": 9,
    "total_timeouts": 0,
    "total_tests": 20,
    "elapsed_sec": 66.28961491584778
  },
  "trained": {
    "model_name": "trained",
    "model_ref": "C:\\Projects\\TgSummarizer\\src\\prolog_merged",
    "tasks": 10,
    "pass_rate": 0.2,
    "syntax_pass_rate": 1.0,
    "timeout_rate": 0.0,
    "passed_all": 2,
    "syntax_ok": 10,
    "total_timeouts": 0,
    "total_tests": 20,
    "elapsed_sec": 28.317045211791992
  }
}

{'task_id': 'family_grandparent', 'base_pass': False, 'trained_pass': False, 'base_tests': '1/2', 'trained_tests': '1/2', 'base_syntax': True, 'trained_syntax': True}
{'task_id': 'my_member', 'base_pass': False, 'trained_pass': False, 'base_tests': '1/2', 'trained_tests': '0/2', 'base_syntax': True, 'trained_syntax': True}
{'task_id': 'list_last', 'base_pass'

In [12]:
artifact = {
    "comparison": comparison,
    "per_task": per_task,
    "base_failed_results": base_failed_results,
    "trained_failed_results": trained_failed_results,
    "base_only_failures": base_only_failures,
    "base_results": base_results,
    "trained_results": trained_results,
}

artifact_path = OUTPUT_DIR / "base_vs_trained_results.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
artifact_path

WindowsPath('C:/Projects/TgSummarizer/outputs/model_benchmark/base_vs_trained_results.json')